Create External table with liquid clustering

In [0]:
%sql
drop table if exists nyc_taxi;
create table nyc_taxi (
   vendor_id	string,
pickup_datetime	timestamp,
dropoff_datetime	timestamp,
passenger_count	int,
trip_distance	double,
pickup_longitude	double,
pickup_latitude	double,
rate_code_id	int,
store_and_fwd_flag	string,
dropoff_longitude	double,
dropoff_latitude	double,
payment_type	string,
fare_amount	double,
extra	double,
mta_tax	double,
tip_amount	double,
tolls_amount	double,
total_amount	double
) using delta
cluster by (vendor_id ,trip_distance)
tblproperties (
delta.autoOptimize.optimizeWrite = false,
delta.autoOptimize.autoCompact = false
)
LOCATION 'abfss://extmetastore@externalmetastore.dfs.core.windows.net/cat1'

In [0]:
file_path = 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow'
all_files = [f.path for f in dbutils.fs.ls(file_path) if f.path.endswith('parquet')]

files10 = all_files[:5]
df = spark.read.parquet(*files10)
df1 = df.repartition(200)

df1.write.format('delta').mode('overwrite').saveAsTable("nyc_taxi")

Though we had 200 partitions, due to clustering enabled, it created only 14 files

In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,0.899,part-00006-471d8827-f5fa-4ffe-9076-848a8976ebf7.c000.zstd.parquet
0.0,0.7,part-00001-b6f995f1-91d3-4fd9-b495-dc23a5b33ea9.c000.zstd.parquet
0.0,1.7,part-00011-48517b74-4975-4603-b5cf-ca9f99fdec71.c000.zstd.parquet
0.8,1.1,part-00009-7243fd5c-d5d8-44ea-9e05-3baf9b4d67a9.c000.zstd.parquet
0.9,1.39,part-00002-dd3c4474-58d7-4322-a590-7e6b8197c8c7.c000.zstd.parquet
1.2,1.7,part-00012-f09a11f1-e993-4e4d-bf5c-f8c099edc01b.c000.zstd.parquet
1.4,1.994,part-00007-866f1b28-1f5a-48f1-995c-d25a96f3c39e.c000.zstd.parquet
1.8,2.6,part-00008-e41fae91-e72a-48f3-8bb4-6fa7ecbf2c68.c000.zstd.parquet
1.8,48.7,part-00000-8782b2f5-17aa-42a5-916a-66a8f3f4c4be.c000.zstd.parquet
2.0,2.92,part-00010-dbf5ea0d-b954-4d6e-9daf-05407462238e.c000.zstd.parquet


In [0]:
%sql
describe history nyc_taxi

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-31T16:35:05.000Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [""vendor_id"",""trip_distance""], description -> null, isManaged -> false, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""databricks.delta.expressionStats.selectedColumns"":""upper(vendor_id),lower(vendor_id)"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-9caf5483-2efa-43de-9b08-a97d861298e3"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e7c66028-be52-4a98-89bb-de74175d1411""}, statsOnLoad -> false, clusteringOnWriteStatus -> late-stage clustering triggered)",null,List(3799693965887743),f8fc038b-074d-4609-8290-42ab6a9cdf99,0831-144423-i1ydcey7-v2n,0,WriteSerializable,false,"Map(numFiles -> 14, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 45142478, numOutputBytes -> 1019605695)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-31T16:34:30.000Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [""vendor_id"",""trip_distance""], description -> null, isManaged -> false, properties -> {""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""databricks.delta.expressionStats.selectedColumns"":""upper(vendor_id),lower(vendor_id)"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-9caf5483-2efa-43de-9b08-a97d861298e3"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e7c66028-be52-4a98-89bb-de74175d1411""}, statsOnLoad -> false)",null,List(3799693965887743),0883d475-39ee-47e3-bbc9-fc00aa75138a,0831-144423-i1ydcey7-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13


In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,1.7,part-00011-48517b74-4975-4603-b5cf-ca9f99fdec71.c000.zstd.parquet
0.0,0.899,part-00006-471d8827-f5fa-4ffe-9076-848a8976ebf7.c000.zstd.parquet
0.0,0.7,part-00001-b6f995f1-91d3-4fd9-b495-dc23a5b33ea9.c000.zstd.parquet
0.8,1.1,part-00009-7243fd5c-d5d8-44ea-9e05-3baf9b4d67a9.c000.zstd.parquet
0.9,1.39,part-00002-dd3c4474-58d7-4322-a590-7e6b8197c8c7.c000.zstd.parquet
1.2,1.7,part-00012-f09a11f1-e993-4e4d-bf5c-f8c099edc01b.c000.zstd.parquet
1.4,1.994,part-00007-866f1b28-1f5a-48f1-995c-d25a96f3c39e.c000.zstd.parquet
1.8,2.6,part-00008-e41fae91-e72a-48f3-8bb4-6fa7ecbf2c68.c000.zstd.parquet
1.8,48.7,part-00000-8782b2f5-17aa-42a5-916a-66a8f3f4c4be.c000.zstd.parquet
2.0,2.92,part-00010-dbf5ea0d-b954-4d6e-9daf-05407462238e.c000.zstd.parquet


Auto mode allowed on managed table only

In [0]:
%sql
alter table nyc_taxi cluster by auto;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5227756387743547>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'alter table nyc_taxi cluster by auto;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, line, cell)
    210 except BaseException as e:
    211     self.driver_activity_logger.